# Spike: which months changed, from Iceberg's snapshots alone

Incremental runs learn what changed from the API's write records (`dataset_writes`). The question
here: can they learn it from the tables themselves? Each snapshot lists the data files it added
and removed, each file has its column bounds, so the months a snapshot touched are the union of
its files' `value_time` ranges. If that matches the write records (and catches writes they miss),
any writer (API, job, Spark, a Prefect flow) is visible to incremental runs.

Read-only: this reads table metadata (snapshots, manifests) as you, and the API's write records.

Runs in JupyterHub, on KinD or use1. Set `TABLES` below; section 2 is for the controlled KinD test.

In [ ]:
%pip install -q "pyiceberg>=0.10" pandas httpx

In [ ]:
import os
import time
from collections import defaultdict

import httpx
import pandas as pd
from pyiceberg.catalog import load_catalog
from pyiceberg.conversions import from_bytes
from pyiceberg.manifest import ManifestContent, ManifestEntryStatus

POLARIS = os.environ.get(
    "POLARIS_URL", "http://polaris.teehr-hub.svc.cluster.local:8181"
).rstrip("/")
REALM = os.environ.get("POLARIS_DEFAULT_REALM", "teehr")
REGION = os.environ.get("AWS_REGION", "us-east-1")
S3_ENDPOINT = os.environ.get("REMOTE_CATALOG_S3_ENDPOINT")  # KinD only (RustFS)
CONTROL_PLANE = os.environ.get(
    "CONTROL_PLANE_URL", "http://teehr-control-plane.teehr-hub.svc.cluster.local:8000"
).rstrip("/")

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)


def access_token():
    """Your access token, refreshed when it's about to expire (JupyterHub sets it)."""
    try:
        from teehr.utils.keycloak_auth import refresh_user_token_from_env
    except ImportError:
        return os.environ["POLARIS_USER_TOKEN"]
    access, _refresh = refresh_user_token_from_env()
    return access


def catalog():
    props = {
        "type": "rest",
        "uri": f"{POLARIS}/api/catalog",
        "warehouse": REALM,
        "token": access_token(),
        "header.X-Polaris-Realm": REALM,
        "header.X-Iceberg-Access-Delegation": "vended-credentials",
        "s3.region": REGION,
    }
    if S3_ENDPOINT:
        props |= {"s3.endpoint": S3_ENDPOINT, "s3.path-style-access": "true"}
    return load_catalog("polaris", **props)


cat = catalog()

## Changed months from snapshots

For each snapshot after `since` (exclusive): skip `replace` (a rewrite that changes no rows);
otherwise take the entries the snapshot itself added or deleted, in the manifests it wrote, and
their `value_time` bounds. Delete files (merge-on-read) can't be bounded by `value_time`: they're
flagged, not guessed.

In [ ]:
def _month_range(lo, hi):
    out = []
    m = pd.Timestamp(lo).tz_convert(None).to_period("M")
    end = pd.Timestamp(hi).tz_convert(None).to_period("M")
    while m <= end:
        out.append(str(m))
        m += 1
    return out


def snapshot_changes(table, column="value_time"):
    """Per snapshot (oldest first): operation, files and rows added/removed, the months its
    files' column bounds span, whether delete files were involved, and how long it took."""
    schema = table.schema()
    field = None
    try:
        field = schema.find_field(column)
    except ValueError:
        pass
    snaps = sorted(table.metadata.snapshots, key=lambda s: s.sequence_number or 0)
    rows = []
    for snap in snaps:
        started = time.monotonic()
        op = snap.summary.operation.value if snap.summary else "?"
        summary = dict(snap.summary.additional_properties) if snap.summary else {}
        months, added, removed, delete_files = set(), 0, 0, 0
        if op != "replace":
            for manifest in snap.manifests(table.io):
                if manifest.added_snapshot_id != snap.snapshot_id:
                    continue  # written by an earlier snapshot: not this one's changes
                if manifest.content == ManifestContent.DELETES:
                    delete_files += 1
                    continue
                for entry in manifest.fetch_manifest_entry(table.io, discard_deleted=False):
                    if entry.snapshot_id != snap.snapshot_id:
                        continue
                    if entry.status == ManifestEntryStatus.ADDED:
                        added += 1
                    elif entry.status == ManifestEntryStatus.DELETED:
                        removed += 1
                    else:
                        continue
                    if field is None:
                        continue
                    lo = (entry.data_file.lower_bounds or {}).get(field.field_id)
                    hi = (entry.data_file.upper_bounds or {}).get(field.field_id)
                    if lo is None or hi is None:
                        months.add("<no bounds>")
                        continue
                    lo = pd.Timestamp(from_bytes(field.field_type, lo), unit="us", tz="UTC")
                    hi = pd.Timestamp(from_bytes(field.field_type, hi), unit="us", tz="UTC")
                    months.update(_month_range(lo, hi))
        rows.append({
            "snapshot_id": snap.snapshot_id,
            "at": pd.Timestamp(snap.timestamp_ms, unit="ms", tz="UTC").floor("s"),
            "operation": op,
            "files_added": added,
            "files_removed": removed,
            "rows_added": summary.get("added-records"),
            "rows_deleted": summary.get("deleted-records"),
            "delete_files": delete_files,
            "months": sorted(months) if field is not None else "(no value_time: changed)",
            "seconds": round(time.monotonic() - started, 2),
        })
    return pd.DataFrame(rows)


def write_records(project, dataset):
    """The API's write records for the dataset, by snapshot (as you)."""
    resp = httpx.get(
        f"{CONTROL_PLANE}/v1/projects/{project}/datasets/{dataset}/writes",
        params={"limit": 1000},
        headers={"Authorization": f"Bearer {access_token()}"},
        timeout=60,
    )
    if resp.status_code != 200:
        print(f"write records: HTTP {resp.status_code} {resp.text[:200]}")
        return {}
    return {int(w["snapshot_id"]): w for w in resp.json() if w.get("snapshot_id")}


def compare(project, dataset, column="value_time"):
    """Snapshots grouped by the write that made them (one write can commit several, e.g. a
    delete and an append; its record names the last), each group's months against the
    record's."""
    table = cat.load_table((project, dataset))
    began = time.monotonic()
    changes = snapshot_changes(table, column)
    took = time.monotonic() - began
    records = write_records(project, dataset)
    groups, pending = [], []
    for row in changes.itertuples():
        if row.operation == "replace":
            groups.append({"snapshots": 1, "at": row.at, "operations": "replace",
                           "months": [], "recorded_months": None, "verdict": "replace: ignored"})
            continue
        pending.append(row)
        if row.snapshot_id in records:
            groups.append(_group(pending, records[row.snapshot_id]))
            pending = []
    if pending:
        groups.append(_group(pending, None))
    out = pd.DataFrame(groups)
    print(f"{project}.{dataset}: {len(changes)} snapshots, {len(records)} write records, "
          f"metadata read in {took:.1f}s")
    return out


def _n(value):
    return int(value) if pd.notna(value) else 0


def _group(rows, record):
    months = set()
    for r in rows:
        if isinstance(r.months, list):
            months.update(r.months)
    snap = sorted(months)
    rec = record.get("months") if record else None
    if record is None:
        verdict = "NOT IN WRITE RECORDS"
    elif rec is None:
        verdict = "recorded (no months: overwrite or not a timeseries)"
    elif set(snap) == set(rec):
        verdict = "same months"
    elif set(rec) <= set(snap):
        verdict = f"snapshots +{len(set(snap) - set(rec))} month(s)"
    else:
        verdict = f"MISSED {sorted(set(rec) - set(snap))}"
    return {"snapshots": len(rows), "at": rows[-1].at,
            "operations": "+".join(r.operation for r in rows),
            "rows_added": sum(_n(r.rows_added) for r in rows),
            "rows_deleted": sum(_n(r.rows_deleted) for r in rows),
            "months": snap if rows[0].months != "(no value_time: changed)" else "(changed)",
            "recorded_months": rec, "verdict": verdict}


## 1. Real tables

On use1: `usgs_streamflow` (copied in with Spark, then sorted rewrites), `nwm30_medium_range`
(forecasts, partitioned by `reference_time`; each reaches ~10 days past its issue), `usgs_nwm30`
(a crosswalk: no `value_time`). On KinD, the `nwm31` names.

In [ ]:
ON_KIND = bool(S3_ENDPOINT)
TABLES = (
    [("public", "usgs_streamflow"), ("public", "nwm31_short_range"), ("public", "usgs_nwm31")]
    if ON_KIND
    else [("public", "usgs_streamflow"), ("public", "nwm30_medium_range"), ("public", "usgs_nwm30")]
)
results = {}
for project, dataset in TABLES:
    results[dataset] = compare(project, dataset)
    display(results[dataset].tail(15))

### The forecast: value_time months vs partition months

A forecast issued late in a month reaches into the next; the snapshot months (from file bounds)
should include it where the partition (reference_time month) alone wouldn't.

In [ ]:
forecast = TABLES[1][1]
fc = cat.load_table(("public", forecast))
print(fc.spec())
partitions = fc.inspect.partitions().to_pandas()
print("partitions (reference_time months):", len(partitions))
print("value_time months found across snapshots:",
      sorted({m for ms in results[forecast]["months"] if isinstance(ms, list) for m in ms})[-6:])

## 2. Controlled writes (KinD)

Claude writes a scripted sequence to `user_test.cli_spike` through the API (an upload of January,
an upsert overlapping into February, a filtered overwrite of part of January, a failed load, a far
month). Run this after; every snapshot should have a write record, and the months should agree.

In [ ]:
if ON_KIND:
    spike = compare("user_test", "cli_spike")
    display(spike)

## Summary to paste back

In [ ]:
summary = pd.concat(
    {name: df["verdict"].value_counts() for name, df in
     {**results, **({"cli_spike": spike} if ON_KIND else {})}.items()},
    axis=1,
).fillna(0).astype(int)
print(summary.to_string())
# Delete files (merge-on-read) can't be bounded by value_time: count any.
for project, dataset in TABLES:
    raw = snapshot_changes(cat.load_table((project, dataset)))
    print(dataset, "snapshots:", len(raw), "with delete files:", int((raw["delete_files"] > 0).sum()),
          "metadata seconds:", round(raw["seconds"].sum(), 1), "max per snapshot:", raw["seconds"].max())